# Knowledge Agent: Hierarchical Routing Tree (SQL Base)

Dieses Jupyter Notebook initialisiert und konfiguriert das relationale Datenfundament für den **Knowledge Agent**. 

### Zweck der Architektur:
* **Hierarchisches Routing (State-Graph / Decision Tree):** Der Agent nutzt diese SQLite-Datenbank wie einen Entscheidungsbaum (ähnlich einem Random Forest), um eingehende Anfragen schrittweise zu analysieren und über Verzweigungen (Knoten) den exakten Ausführungspfad zu bestimmen.
* **Selbstdokumentation:** Die Struktur ist so ausgelegt, dass das LLM beim Start seine eigenen Navigationsregeln und Zustände ausliest.

## Architektur-Übersicht der Datenbank

Die Datenbank gliedert sich in drei zentrale Komponenten:

1. **`agent_system_manifest`**: Enthält die grundlegenden Systemanweisungen und Direktiven. Das LLM liest diese Tabelle zuerst aus, um zu verstehen, wie es den Routing-Baum zu navigieren hat.
2. **`agent_routing_tree`**: Der eigentliche Entscheidungsbaum (`node_id`, `parent_node_id`, `routing_condition`). Hier sind die logischen Verknüpfungen hinterlegt, welcher Schritt (A -> B -> C) als Nächstes ausgeführt wird.
3. **`agent_execution_memory`**: Dient als Feedback-Loop und Erfahrungsspeicher (Self-Reflection), um erfolgreiche Pfade zu gewichten und den Baum dynamisch zu erweitern.

In [1]:
# Import
import sqlite3
import os

In [2]:
# Globale Definitionen für die Ordnerstruktur
BASE_DIR = "Knowledge"
AGENT_SUBDIR = "knowledge_agent_hierarchical_routing_tree_sql"
# globalisierte Variablen der SQLite-Datenbankdatei
DB_FILENAME = "knowledge_agent_routing_tree.db"

In [3]:
# def für die Initialisierung der Ordnerstruktur
def initialize_find_folder():
    """
    Diese Funktion prüft, ob die notwendige Ordnerstruktur für den Knowledge Agent im Projekt vorhanden ist.
    Gibt bei jedem Teilschritt ein klares Status-Print in der Konsole aus.
    """
    print("--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---")

    # Schritt 1: Hauptverzeichnis "Knowledge" suchen oder erstellen
    print(f"Suche nach Hauptverzeichnis: '{BASE_DIR}'...")
    if not os.path.exists(BASE_DIR):
        os.makedirs(BASE_DIR)
        print(f"--> [ERFOLG] Hauptverzeichnis '{BASE_DIR}' wurde neu erstellt.")
    else:
        print(f"--> [INFO] Hauptverzeichnis '{BASE_DIR}' wurde gefunden.")

    # Schritt 2: Unterordner für den Agenten im Knowledge-Verzeichnis suchen oder erstellen
    target_directory = os.path.join(BASE_DIR, AGENT_SUBDIR)
    print(f"Suche nach Unterordner: '{target_directory}'...")
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
        print(f"--> [ERFOLG] Unterordner '{AGENT_SUBDIR}' wurde neu erstellt.")
    else:
        print(f"--> [INFO] Unterordner '{AGENT_SUBDIR}' existiert bereits.")

    # Schritt 3: Vollständigen Pfad zur Datenbank zusammenbauen
    db_path = os.path.join(target_directory, DB_FILENAME)
    print(f"Pfad-Zusammenführung abgeschlossen. Zieldatei: '{db_path}'")
    print("--- [ENDE] Ordnerstruktur erfolgreich geprüft ---")
    
    return db_path

## Globale Schema-Definitionen & Tabellen-Konstanten

Bevor wir die Datenbanktabellen erstellen, definieren wir alle Tabellennamen und Spalten als globale Konstanten. Dadurch verhindern wir Tippfehler und machen den Code absolut übersichtlich.

* **Tabelle 1 (`agent_system_manifest`)**: Das Start-Manifest für das LLM (liest es zuerst aus, um seine Regeln zu verstehen).
  * Spalte 1: `directive_key` (Primärschlüssel, z. B. `core_logic`)
  * Spalte 2: `explanation_for_agent` (Die Anweisung/Erklärung für das Modell)

In [1]:
# GLOBALE VARIABLEN & SCHEMA-DEFINITIONEN: TABELLE 1
# Name der ersten Tabelle für das System-Manifest (Bedienungsanleitung für das LLM)
TABLE_MANIFEST = "agent_system_manifest"

# Globale Spaltennamen für diese Tabelle
COL_MANIFEST_KEY = "directive_key"          # Der eindeutige Schlüssel (Primary Key, z.B. 'core_logic')
COL_MANIFEST_VAL = "explanation_for_agent"    # Die eigentliche Anweisung / Erklärung für das LLM

In [5]:
# Erstellt agent_system_manifest Tabelle in der SQLite-Datenbank
def create_system_manifest_table(db_path):
    """
    Erstellt die erste Tabelle ('agent_system_manifest') in der SQLite-Datenbank.
    Diese Tabelle dient dem LLM als erster Lesezugriff, um die Architektur zu verstehen 
    und zu wissen, dass die Ausführung universell bei der Start-ID '000' beginnt.
    """
    print(f"--- [START] Erstelle Tabelle '{TABLE_MANIFEST}' ---")
    
    # 1. Verbindung zur SQLite-Datenbank herstellen (unter Verwendung des übergebenen Pfads)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"--> [INFO] Verbindung zur Datenbank geöffnet: {db_path}")

    # 2. Tabelle erstellen unter Nutzung der globalen Konstanten
    create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {TABLE_MANIFEST} (
            {COL_MANIFEST_KEY} TEXT PRIMARY KEY,
            {COL_MANIFEST_VAL} TEXT NOT NULL
        )
    """
    cursor.execute(create_table_query)
    print(f"--> [ERFOLG] Tabelle '{TABLE_MANIFEST}' wurde erfolgreich angelegt (oder war bereits vorhanden).")

    # 3. Globale Direktiven für das LLM definieren (Standard-Konzept für Agenten-Manifeste)
    # Beinhaltet die Kernlogik und den universellen Startpunkt '000'
    initial_directives = [
        ("core_logic", "This database is a hierarchical routing tree. Always start execution at node_id '000'."),
        ("navigation_rule", "Evaluate 'routing_condition' against current user intent, then follow the mapped node."),
        ("self_expansion", "Log all execution steps in 'agent_execution_memory' and dynamically append new branches on success.")
    ]

    # 4. Daten sicher in die Tabelle schreiben (INSERT OR REPLACE verhindert Duplikate)
    insert_query = f"""
        INSERT OR REPLACE INTO {TABLE_MANIFEST} ({COL_MANIFEST_KEY}, {COL_MANIFEST_VAL}) 
        VALUES (?, ?)
    """
    cursor.executemany(insert_query, initial_directives)
    print("--> [ERFOLG] Initiale LLM-Direktiven (inkl. Start-Verweis '000') wurden eingefügt.")

    # 5. Transaktion speichern und Verbindung schließen
    conn.commit()
    conn.close()
    print(f"--- [ENDE] Tabelle '{TABLE_MANIFEST}' erfolgreich initialisiert ---")

## Globale Schema-Definitionen & Tabellen-Konstanten: Tabelle 2 (Routing-Baum)

Tabelle 2 (`agent_routing_tree`) bildet den eigentlichen Entscheidungsbaum ab. 
* `node_id`: Die eindeutige ID des Knotens (z. B. `'000'` für den Start, `'001'` für den nächsten Schritt).
* `parent_node_id`: Der Verweis auf den vorherigen Knoten (um den Pfad zurückzuverfolgen).
* `routing_condition`: Die Bedingung oder Absicht, wann dieser Pfad gewählt wird.
* `action_payload`: Die auszuführende Aktion oder das Skript.
* `target_table`: Eventuelle Detail-Tabellen für fachspezifische Daten.
* `success_weight`: Gewichtung/Erfahrungswert des Pfades.

In [ ]:
TABLE_ROUTING = "agent_routing_tree"

ROUTING_COLUMNS = {
    "node_id": "TEXT PRIMARY KEY",
    "parent_node_id": "TEXT",
    "node_purpose": "TEXT NOT NULL",
    "routing_condition": "TEXT NOT NULL",
    "action_payload": "TEXT NOT NULL",
    "target_table": "TEXT",
    "success_weight": "REAL DEFAULT 1.0",
    "execution_count": "INTEGER DEFAULT 0",
    "is_active": "INTEGER DEFAULT 1",
    "year": "INTEGER",
    "month": "INTEGER",
    "day": "INTEGER",
    "hour": "INTEGER",
    "minute": "INTEGER",
    "second": "INTEGER"
}

In [ ]:
def create_routing_tree_table(db_path):
    print(f"--- [START] Prüfe und initialisiere leere Tabelle '{TABLE_ROUTING}' ---")
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"--> [INFO] Verbindung zur Datenbank geöffnet: {db_path}")

    # Schritt 1: Prüfen, ob die Tabelle überhaupt existiert
    cursor.execute(f"""
        SELECT name FROM sqlite_master WHERE type='table' AND name='{TABLE_ROUTING}';
    """)
    table_exists = cursor.fetchone()

    if not table_exists:
        print(f"--> [INFO] Tabelle '{TABLE_ROUTING}' existiert noch nicht. Wird komplett neu erstellt...")
        columns_definition = ", ".join([f"{col_name} {col_type}" for col_name, col_type in ROUTING_COLUMNS.items()])
        create_table_query = f"""
            CREATE TABLE IF NOT EXISTS {TABLE_ROUTING} (
                {columns_definition}
            )
        """
        cursor.execute(create_table_query)
        print(f"--> [ERFOLG] Tabelle '{TABLE_ROUTING}' wurde erfolgreich und leer neu erstellt.")
    else:
        print(f"--> [INFO] Tabelle '{TABLE_ROUTING}' existiert bereits. Prüfe auf fehlende Spalten...")
        
        # Bestehende Spalten in der Datenbank auslesen
        cursor.execute(f"PRAGMA table_info({TABLE_ROUTING});")
        existing_columns_info = cursor.fetchall()
        existing_column_names = [col[1] for col in existing_columns_info]

        # Abgleich: Welche Spalten aus unserem Schema fehlen in der Datenbank?
        for col_name, col_type in ROUTING_COLUMNS.items():
            if col_name not in existing_column_names:
                print(f"--> [INFO] Spalte '{col_name}' fehlt in der Tabelle. Füge sie hinzu...")
                alter_query = f"ALTER TABLE {TABLE_ROUTING} ADD COLUMN {col_name} {col_type};"
                try:
                    cursor.execute(alter_query)
                    print(f"--> [ERFOLG] Spalte '{col_name}' erfolgreich hinzugefügt.")
                except Exception as e:
                    print(f"--> [FEHLER] Konnte Spalte '{col_name}' nicht hinzufügen: {e}")
            else:
                print(f"--> [OK] Spalte '{col_name}' ist bereits vorhanden.")

    conn.commit()
    conn.close()
    print(f"--- [ENDE] Leere Tabellen- und Spaltenprüfung für '{TABLE_ROUTING}' abgeschlossen ---")

## Initialisierung der SQLite-Datenbank

Im folgenden Code-Block wird die Verbindung zur lokalen SQLite-Datenbank hergestellt und die oben beschriebene Tabellenstruktur fehlerfrei aufgebaut. Falls die Datenbank noch nicht existiert, wird sie automatisch generiert.